In [2]:
import os
os.chdir('/data2/qingyuan/projects/zengcheng3')
from reev_control.envs.trajectory_loader import TrajectoryLoader

import pandas as pd

In [3]:
data_folder = 'data/train/REEV07RearDrive_Mar2025'
files = os.listdir(data_folder)
files[0]

'20250317_RearDrive_c988d316cf1639445e40ec178b5cb705.csv'

In [ ]:
sample = pd.read_csv(os.path.join(data_folder, files[500]))
print(len(sample))
sample.head()

5913


,eventId,date_time,EspVehSpd,EspLgtAccel,VcuVehAvrgEgyCnseDrivingCyc,VcuVehAvrgEgyCnseLongTime,VcuVehAvrgEgyCnseIn10km,VcuVehAvrgEgyCnseIn25km,DcdcCnseActPwr,TmsFrntWindLvl,...,EmsAltiFac_original,EmsAltiFac,BcuBattTMax,BcuBattTMin,BcuBattIntrHeatReq_heating,BcuBattIntrHeatReq_cooling,BcuBattIntrHeatReq_no,VcuRealAccPedl,CdcTotMilg,CdcDrvCnseDrivingCyc
0,2ce80d2ab91fecbcc010131df4499a95,2025-03-19 17:49:49,3.48750,-0.21,NaN,17.0,14.1,18.6,0.621,0.0,...,1.0,1.0,19.0,18.0,0,0,1,0.0,3290.2,0.0
1,2ce80d2ab91fecbcc010131df4499a95,2025-03-19 17:49:50,4.72500,0.29,NaN,17.0,14.1,18.6,0.610,0.0,...,1.0,1.0,19.0,18.0,0,0,1,0.0,3290.2,0.0
2,2ce80d2ab91fecbcc010131df4499a95,2025-03-19 17:49:51,4.05000,-0.09,NaN,17.0,14.1,18.6,0.604,0.0,...,1.0,1.0,19.0,18.0,0,0,1,14.3,3290.2,0.0
3,2ce80d2ab91fecbcc010131df4499a95,2025-03-19 17:49:52,7.14375,0.53,NaN,17.0,14.1,18.6,0.595,0.0,...,1.0,1.0,19.0,18.0,0,0,1,14.5,3290.2,0.0
4,2ce80d2ab91fecbcc010131df4499a95,2025-03-19 17:49:53,8.15625,0.34,NaN,17.0,14.1,18.6,0.591,0.0,...,1.0,1.0,19.0,18.0,0,0,1,9.3,3290.2,0.0


In [6]:
sample['CdcTotMilg'].iloc[-1] - sample['CdcTotMilg'].iloc[0]

70.70000000000027

In [8]:
import os
import pandas as pd
import pickle


def filter_valid_files(file_list, data_folder):
    valid_files = []
    for file_name in file_list:
        path = os.path.join(data_folder, file_name)
        try:
            df = pd.read_csv(path)
            if (
                (df['CdcTotMilg'].iloc[-1] - df['CdcTotMilg'].iloc[0] >= 100) and
                (len(df) >= 3600) and
                (df['EspVehSpd'].mean() >= 35)
            ):
                valid_files.append(file_name)
        except Exception as e:
            print(f"Skipping {file_name} due to error: {e}")
    return valid_files

data_folder = 'data/train/REEV07RearDrive_Mar2025'
file_list = [os.path.join(data_folder, f) for f in os.listdir(data_folder) if f.endswith(".csv")]
filtered = filter_valid_files(files, data_folder)

print(len(file_list))
print(len(filtered))
with open('filtered_files.pkl', 'wb') as f:
    pickle.dump(filtered, f)


79876
19300


In [2]:
"a.csv".split('.')[0]

'a'

In [3]:
import os
os.chdir('/data2/qingyuan/projects/zengcheng3')
import pandas as pd
from stable_baselines3.common.vec_env import DummyVecEnv, VecNormalize
from stable_baselines3.common.monitor import Monitor

from reev_control.envs import SimpleVehicleEnv2, SimpleVehicleEnv3
from reev_control.envs.wrappers import ActionFlatteningWrapper, InfoSumWrapper
from reev_control.custom_ppo import CustomPPO  # your CustomPPO
import torch
import yaml

# --- Settings ---
folder = "train_results/wandb/offline-run-20250603_171003-2v1jr8zy/files"
model_path = f"{folder}/model.zip"
vecnorm_path = f"{folder}/vec_env.pkl"  # adjust path

# load wandb config from train folder
with open(f'{folder}/config.yaml', 'r') as file:
    train_config = yaml.safe_load(file)

train_config = {
    k: v["value"]
    for k, v in train_config.items()
    if (isinstance(v, dict) and "value" in v and k!='_wandb')
}

# obs_keys_to_log = ["some_obs_name1", "some_obs_name2"]  # update based on your actual obs
# info_keys_to_log = ["nvh_reward_sum", "efficiency_reward_sum", "step_soc_reward_sum", "end_soc_reward"]  # update accordingly

# --- Create single environment for inference ---
def make_eval_env(seed: int = 0, **kwargs):

    def _init():

        env = SimpleVehicleEnv2(
            data_folder='data/train/REEV07RearDrive_Mar2025',
            seed=seed, **kwargs)
        # env.reset()
        return env

    return _init

env = DummyVecEnv([make_eval_env(**train_config)])




# --- Load VecNormalize ---
env = VecNormalize.load(vecnorm_path, env)
env.training = False
env.norm_reward = False     # want original reward in eval

In [9]:
env.unwrapped.unwrapped.unwrapped